# 🎨 PhotoEnhance — AI Photo Upscaler
**Безкоштовне покращення якості фото за допомогою Real-ESRGAN**

---

| Параметр | Значення |
|---|---|
| Модель | Real-ESRGAN |
| GPU | Google Colab T4 (16GB) |
| Масштаб | x2, x4, x8 |
| Формати | JPG, PNG, WEBP |
| Вартість | $0 |

> ⚠️ **Переконайся, що увімкнено GPU:** Runtime → Change runtime type → T4 GPU

## Як користуватись: 3 кроки
| Крок | Що робить |
|---|---|
| ▶ **Крок 1** | Встановлення, KeepAlive, перевірка GPU |
| ▶ **Крок 2** | Google Drive + завантаження фото |
| ▶ **Крок 3** | Обробка та скачування результату |


In [ ]:
# @title ⚙️ Крок 1 — Встановлення та перевірка { display-mode: "form" }
# @markdown Запускай **один раз** на початку кожної Colab-сесії.
# @markdown Повторний запуск безпечний — все вже встановлене буде пропущено.

# ══════════════════════════════════════════════════════════════
# STUDENT GPU TIPS (безкоштовно або майже безкоштовно):
#
#  ☁️  Google Colab       — T4 безкоштовно | A100 через Colab Pro
#  🏆  Kaggle Notebooks   — T4/P100, 30 год/тиждень, безкоштовно
#      (File → Settings → Accelerator → GPU T4x2)
#  ⚡  Lightning.ai       — A10G, безкоштовний tier
#  🎓  GitHub Education   — Student Pack → AWS/Azure/GCP кредити
#      (github.com/education, підтвердь email університету)
#  🚀  Vast.ai            — оренда GPUs ~$0.2/год (Colab Pro дешевше)
# ══════════════════════════════════════════════════════════════

import sys
import os
import subprocess
import threading

import ipywidgets as widgets
from IPython.display import display, Javascript, HTML

# ─── прогрес-бар ────────────────────────────────────────────────────
_pb = widgets.IntProgress(
    value=0, min=0, max=100,
    description='⏳  0%', bar_style='info',
    style={'bar_color': '#4a90e2', 'description_width': '65px'},
    layout=widgets.Layout(width='100%', height='28px'),
)
_lbl = widgets.HTML('<span style="font-size:12px;color:#555;">Перевірка середовища...</span>')
display(widgets.VBox([_pb, _lbl]))

# ════════════════════════════════════════════════
# §0  ПЕРЕВІРКИ СЕРЕДОВИЩА
# ════════════════════════════════════════════════
def _check(cond, ok_msg, fail_msg, fatal=False):
    if cond:
        print(f'✅ {ok_msg}')
    else:
        print(f'{"❌" if fatal else "⚠️"} {fail_msg}')
    return cond

_ready = True

# Python версія
_v = sys.version_info
_ready &= _check(
    _v >= (3, 8),
    f'Python {_v.major}.{_v.minor}.{_v.micro}',
    f'Python {_v.major}.{_v.minor} — потрібен 3.8+.',
    fatal=True,
)

# Colab (не зберігаємо модуль — лише перевірка)
try:
    __import__('google.colab')  # Colab env check (side-effect import)
    print('✅ Середовище: Google Colab')
except ImportError:
    print('⚠️  Не Colab — деякі функції (Drive, download) не працюватимуть.')

# RAM
try:
    with open('/proc/meminfo') as _mf:
        _ram_gb = int([l for l in _mf if 'MemAvailable' in l][0].split()[1]) / 1048576
    _check(_ram_gb >= 4,
           f'RAM вільно: {_ram_gb:.1f} GB',
           f'Мало RAM: {_ram_gb:.1f} GB — можливі OOM при ×8.')
except Exception:
    print('ℹ️  RAM: не вдалося перевірити')

# Диск
try:
    import shutil as _sh
    _, _, _fd = _sh.disk_usage('/content')
    _check(_fd / 1073741824 >= 3,
           f'Диск вільно: {_fd / 1073741824:.1f} GB',
           f'Мало місця: {_fd / 1073741824:.1f} GB — можливі помилки запису.')
except Exception:
    print('ℹ️  Диск: не вдалося перевірити')

if not _ready:
    _pb.bar_style = 'danger'
    raise SystemExit('❌ Усунь помилки вище та перезапусти Крок 1.')

_pb.value = 5; _pb.description = '🔎  5%'
_lbl.value = '<span style="font-size:12px;color:#555;">Встановлення залежностей...</span>'
print('\n📦 Починаємо встановлення...\n')

# ════════════════════════════════════════════════
# §1  ЗАЛЕЖНОСТІ
# ════════════════════════════════════════════════
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
os.environ['PIP_NO_WARN_SCRIPT_LOCATION']   = '1'

print('📦 basicsr==1.4.2 / facexlib / gfpgan...')
get_ipython().run_line_magic('pip', 'install -q "basicsr==1.4.2" facexlib gfpgan')  # type: ignore
_pb.value = 18; _pb.description = '⬇️  18%'

print('📦 Pillow / opencv / ipywidgets...')
get_ipython().run_line_magic('pip', 'install -q "Pillow>=9" "opencv-python-headless>=4.8" ipywidgets')  # type: ignore
_pb.value = 23; _pb.description = '⬇️  23%'

# ── torchvision >= 0.16 compat fix (basicsr шукає старий module) ───
from types import ModuleType
import torchvision.transforms.functional as _tvf
_compat = ModuleType('torchvision.transforms.functional_tensor')
_compat.rgb_to_grayscale = _tvf.rgb_to_grayscale  # type: ignore
sys.modules.setdefault('torchvision.transforms.functional_tensor', _compat)
print('✅ torchvision compat patch OK')
_pb.value = 26; _pb.description = '📦  26%'

# ════════════════════════════════════════════════
# §2  Real-ESRGAN
# ════════════════════════════════════════════════
_lbl.value = '<span style="font-size:12px;color:#555;">Клонування Real-ESRGAN...</span>'
if not os.path.isdir('/content/Real-ESRGAN'):
    print('📦 Клонування Real-ESRGAN...')
    get_ipython().system('git clone -q https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN')  # type: ignore
    _pb.value = 33; _pb.description = '📦  33%'
    get_ipython().run_line_magic('pip', 'install -q -r /content/Real-ESRGAN/requirements.txt')  # type: ignore
else:
    print('✅ Real-ESRGAN вже є')
os.chdir('/content/Real-ESRGAN')
# pip install -e є надійнішим за setup.py develop і безпечний для повторного запуску
print('📦 pip install -e Real-ESRGAN...')
get_ipython().run_line_magic('pip', 'install -q -e /content/Real-ESRGAN')  # type: ignore
print('✅ Real-ESRGAN встановлено')
_pb.value = 45; _pb.description = '📦  45%'

# ════════════════════════════════════════════════
# §3  CodeFormer
# ════════════════════════════════════════════════
_lbl.value = '<span style="font-size:12px;color:#555;">Клонування CodeFormer...</span>'
if not os.path.isdir('/content/CodeFormer'):
    print('🔬 Клонування CodeFormer...')
    get_ipython().system('git clone -q https://github.com/sczhou/CodeFormer.git /content/CodeFormer')  # type: ignore
    _pb.value = 52; _pb.description = '🔬  52%'
    os.chdir('/content/CodeFormer')
    get_ipython().run_line_magic('pip', 'install -q -r requirements.txt')  # type: ignore
    get_ipython().system('python basicsr/setup.py develop -q 2>/dev/null')  # type: ignore
    _pb.value = 60; _pb.description = '🔬  60%'
    _lbl.value = '<span style="font-size:12px;color:#555;">Ваги facelib та CodeFormer...</span>'
    get_ipython().system('python scripts/download_pretrained_models.py facelib')  # type: ignore
    _pb.value = 72; _pb.description = '⬇️  72%'
    get_ipython().system('python scripts/download_pretrained_models.py CodeFormer')  # type: ignore
    print('✅ CodeFormer встановлено')
else:
    print('✅ CodeFormer вже є')
_pb.value = 82; _pb.description = '🔬  82%'

# Re-pin basicsr==1.4.2 — requirements.txt може встановити новішу версію (без data.degradations)
print('📦 Re-pin basicsr==1.4.2 (фіналізація)...')
get_ipython().run_line_magic('pip', 'install -q "basicsr==1.4.2"')  # type: ignore
print('✅ basicsr==1.4.2 OK')

# ── Патч: CodeFormer має власний (неповний) basicsr ─────────────────
# /content/CodeFormer стоїть першим у sys.path → Python знаходить
# CodeFormer basicsr замість повного pip basicsr==1.4.2.
#
# ПРАВИЛЬНЕ РІШЕННЯ:
#   1. Копіюємо CodeFormer-специфічні файли (codeformer_arch тощо)
#      у ПОВНИЙ pip basicsr.
#   2. Перейменовуємо /content/CodeFormer/basicsr → basicsr_cf_bak,
#      щоб Python більше не знаходив неповну версію першою.
import sysconfig as _sc
import shutil as _sh2
import glob as _gl

_site    = _sc.get_paths()['purelib']
_pip_bsr = os.path.join(_site, 'basicsr')
if not os.path.isdir(_pip_bsr):
    _found = _gl.glob('/usr/local/lib/python*/dist-packages/basicsr')
    _pip_bsr = _found[0] if _found else ''

_cf_bsr     = '/content/CodeFormer/basicsr'
_cf_bsr_bak = '/content/CodeFormer/basicsr_cf_bak'
_patched    = 0

# Файли, де CF-версія є надмножиною pip-версії (містить додаткові функції)
# → завжди перезаписуємо pip-версію CF-версією
_CF_OVERWRITE = {
    os.path.join('data', 'data_util.py'),
}

if _pip_bsr and os.path.isdir(_pip_bsr) and os.path.isdir(_cf_bsr):
    # Крок 1: копіюємо CF-файли → pip basicsr
    #   - нові файли: завжди копіюємо
    #   - файли з _CF_OVERWRITE: перезаписуємо (CF є надмножиною pip)
    for _root, _dirs, _files in os.walk(_cf_bsr):
        _rel     = os.path.relpath(_root, _cf_bsr)
        _dst_dir = os.path.join(_pip_bsr, _rel)
        os.makedirs(_dst_dir, exist_ok=True)
        for _f in _files:
            if _f.endswith('.pyc'):
                continue
            _rel_f = os.path.join(_rel, _f) if _rel != '.' else _f
            _dst_f = os.path.join(_dst_dir, _f)
            _force = _rel_f in _CF_OVERWRITE
            if not os.path.isfile(_dst_f) or _force:
                _sh2.copy2(os.path.join(_root, _f), _dst_f)
                _patched += 1
    # Крок 2: ховаємо CF basicsr щоб Python не знаходив його першим
    if os.path.isdir(_cf_bsr_bak):
        _sh2.rmtree(_cf_bsr_bak)
    os.rename(_cf_bsr, _cf_bsr_bak)
    print(f'✅ Patch OK: {_patched} CF-файлів → pip basicsr; CF basicsr схований')
elif not os.path.isdir(_cf_bsr):
    # Already hidden — re-apply overwrite files from backup so they are always up to date
    if _pip_bsr and os.path.isdir(_cf_bsr_bak):
        for _rel_f in _CF_OVERWRITE:
            _src_f = os.path.join(_cf_bsr_bak, _rel_f)
            _dst_f = os.path.join(_pip_bsr, _rel_f)
            if os.path.isfile(_src_f):
                _sh2.copy2(_src_f, _dst_f)
                _patched += 1
    if _patched:
        print(f'✅ CF basicsr схований; {_patched} файлів оновлено з бекапу')
    else:
        print('✅ CF basicsr вже схований (нічого не змінилось)')
else:
    print('⚠️  pip basicsr не знайдено — пропускаємо патч')

# Обов'язково: очищаємо sys.modules від basicsr — Python міг
# закешувати CF-версію ще під час pip install. Без цього rename
# не допомагає: модуль залишається старим у пам'яті.
_purged = [k for k in list(sys.modules) if k == 'basicsr' or k.startswith('basicsr.')]
for k in _purged:
    del sys.modules[k]
if _purged:
    print(f'🔄 Purged {len(_purged)} basicsr-модулів з sys.modules → свіжий import з pip')

os.chdir('/content/Real-ESRGAN')
for _p in ['/content/Real-ESRGAN', '/content/CodeFormer']:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ════════════════════════════════════════════════
# §4  KeepAlive
# ════════════════════════════════════════════════
_pb.value = 86; _pb.description = '🔌  86%'
display(Javascript("""
(function() {
    if (window._pe_ka) clearInterval(window._pe_ka);
    window._pe_ka = setInterval(function() {
        fetch('/api/kernels').catch(function(){});
    }, 45000);
    console.log('[PhotoEnhance] KeepAlive started (45s)');
})();
"""))

_ka_stop = threading.Event()
def _ka_heartbeat():
    while not _ka_stop.wait(timeout=50):
        pass  # python-side ping щоб kernel не відключився

threading.Thread(target=_ka_heartbeat, daemon=True, name='PE-KeepAlive').start()
print('🔌 KeepAlive активовано')

# ════════════════════════════════════════════════
# §5  GPU + PyTorch
# ════════════════════════════════════════════════
_pb.value = 90; _pb.description = '🖥️  90%'
_lbl.value = '<span style="font-size:12px;color:#555;">Перевірка GPU...</span>'
import torch

torch.backends.cudnn.benchmark        = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
    print('✅ matmul precision=high (TF32)')
except AttributeError:
    print('ℹ️  matmul TF32 via cudnn (PyTorch < 2.0)')

print('=' * 52)
print('🖥️  GPU ІНФОРМАЦІЯ')
print('=' * 52)

if torch.cuda.is_available():
    _gname = torch.cuda.get_device_name(0)
    _gmem  = torch.cuda.get_device_properties(0).total_memory / 1073741824
    _cc    = torch.cuda.get_device_capability(0)

    _nsmi = subprocess.run(
        ['nvidia-smi', '--query-gpu=memory.free,temperature.gpu',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True,
    )
    _vfree, _temp = '?', '?'
    if _nsmi.returncode == 0:
        _pts = _nsmi.stdout.strip().split(', ')
        if len(_pts) >= 2:
            try:
                _vfree = f'{int(_pts[0]) / 1024:.1f}'
                _temp  = _pts[1]
            except (ValueError, IndexError):
                pass

    print(f'✅ GPU:  {_gname}')
    print(f'💾 VRAM: {_gmem:.1f} GB  |  вільно: {_vfree} GB')
    print(f'🌡️  Temp: {_temp}°C  |  Compute: {_cc[0]}.{_cc[1]}')
    print(f'🔧 CUDA: {torch.version.cuda}  |  PyTorch: {torch.__version__}')
    if _cc[0] >= 8:
        print('🚀 TF32 активний (Ampere/Hopper) — +20% при fp32')
    else:
        print(f'ℹ️  Compute {_cc[0]}.{_cc[1]} — FP16 дає 4–8× прискорення')
    if _gmem < 4:
        print('⚠️  VRAM < 4 GB — при ×8 можливий OOM. Використовуй ×2/×4.')
    print('\n✅ Готово!  Переходь до Кроку 2 ▶')
else:
    print('❌ GPU не знайдено!')
    print('👉 Runtime → Change runtime type → T4 GPU → Save → перезапусти Крок 1')
    print('💡 Kaggle: Settings → Accelerator → GPU T4×2 (безкоштовно 30 год/тиж)')

print('=' * 52)
_pb.value = 100; _pb.description = '✅ 100%'; _pb.bar_style = 'success'
_lbl.value = '<span style="font-size:12px;color:#2ecc71;font-weight:bold;">✅ Крок 1 завершено!</span>'

_html_step1 = (
    '<div style="margin-top:14px;padding:14px 20px;'
    'background:linear-gradient(135deg,#1a1a2e,#16213e);'
    'border-radius:10px;border-left:5px solid #2ecc71;font-family:sans-serif;">'
    '<div style="color:#2ecc71;font-size:18px;font-weight:bold;">✅ Крок 1 завершено!</div>'
    '<div style="color:#ccc;margin-top:6px;font-size:13px;">'
    '📦 Real-ESRGAN ✔&nbsp;&nbsp; 🔬 CodeFormer ✔&nbsp;&nbsp; 🔌 KeepAlive ✔&nbsp;&nbsp;'
    '🖥️ GPU ✔&nbsp;&nbsp; ⚡ TF32/FP16 ✔'
    '</div>'
    '<div style="margin-top:10px;font-size:15px;color:#4a90e2;font-weight:bold;">'
    '↓ &nbsp;Гортай вниз і запускай <span style="color:#f9ca24;">Крок 2</span>'
    '</div></div>'
)
display(HTML(_html_step1))


In [ ]:
# @title 🖼️ Крок 2 — Google Drive + Завантаження фото { display-mode: "form" }
# @markdown Drive необов'язковий — без нього файл скачається напряму в Кроці 3.

import sys
import os
import io
import shutil

import ipywidgets as widgets
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML, clear_output

# ════════════════════════════════════════════════
# §0  Перевірка Кроку 1
# ════════════════════════════════════════════════
if 'torch' not in sys.modules:
    raise SystemExit(
        '❌ Крок 1 не виконано! Запусти Крок 1 і дочекайся зеленої галочки.'
    )

import torch as _t
if _t.cuda.is_available():
    _vram = _t.cuda.get_device_properties(0).total_memory / 1073741824
    print(f'✅ GPU: {_t.cuda.get_device_name(0)}  ({_vram:.1f} GB VRAM)')
    if _vram < 4:
        print('⚠️  VRAM < 4 GB — при ×8 може бути OOM. Використовуй ×2 або ×4.')
else:
    print('⚠️  GPU не знайдено — обробка буде дуже повільною.')
    print('   Runtime → Change runtime type → T4 GPU → Save → перезапусти крок 1.')

# Переконуємось що шлях є до запуску перевірки
for _p in ('/content/Real-ESRGAN', '/content/CodeFormer'):
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

# Страхувальне очищення: якщо basicsr закешований з CF-версії —
# видаляємо, щоб наступний import пішов у pip-версію
_bsr_purge = [k for k in list(sys.modules) if k == 'basicsr' or k.startswith('basicsr.')]
for k in _bsr_purge:
    del sys.modules[k]

for _lib in ('realesrgan', 'basicsr', 'cv2', 'PIL'):
    try:
        __import__(_lib)
        print(f'✅ {_lib}')
    except Exception as _e:
        print(f'❌ {_lib} не встановлено — {_e}\n   Перезапусти Крок 1.')

if os.path.isdir('/content/CodeFormer'):
    print('✅ CodeFormer')
else:
    print('⚠️  CodeFormer не знайдено — портретний режим буде недоступний.')

try:
    _, _, _fd = shutil.disk_usage('/content')
    _fd_gb = _fd / 1073741824
    if _fd_gb < 1:
        print(f'⚠️  Диск вільно: {_fd_gb:.1f} GB — може не вистачити.')
    else:
        print(f'✅ Диск вільно: {_fd_gb:.1f} GB')
except Exception:
    pass

print()

# ════════════════════════════════════════════════
# §1  Google Drive (необов'язково)
# ════════════════════════════════════════════════
TEMP_DIR = '/content/photoenhance_temp'
os.makedirs(TEMP_DIR, exist_ok=True)

if os.path.isdir('/content/drive/MyDrive'):
    OUTPUT_DIR      = '/content/drive/MyDrive/PhotoEnhance_Results'
    DRIVE_AVAILABLE = True
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'✅ Drive вже підключено → {OUTPUT_DIR}')
else:
    try:
        from google.colab import drive as _drv  # type: ignore
        print('🔗 Підключення Google Drive...')
        _drv.mount('/content/drive')
        OUTPUT_DIR      = '/content/drive/MyDrive/PhotoEnhance_Results'
        DRIVE_AVAILABLE = True
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print(f'✅ Drive підключено → {OUTPUT_DIR}')
    except Exception as _e:
        print(f'⚠️  Drive: {_e}\n   Файл скачається напряму у Кроці 3.')
        OUTPUT_DIR      = TEMP_DIR
        DRIVE_AVAILABLE = False

MAX_FILE_MB = 100

# ════════════════════════════════════════════════
# §2  Стилі (адаптив для мобільних)
# ════════════════════════════════════════════════
display(HTML("""
<style>
.pe-info  {font-family:monospace;background:#f0f4ff;color:#1a1a2e;padding:10px;border-radius:6px;
           border-left:4px solid #4a90e2;font-size:13px;word-break:break-word;}
.pe-error {font-family:sans-serif;background:#fff0f0;padding:10px;border-radius:6px;
           border-left:4px solid #e74c3c;font-size:13px;}
@media(max-width:600px){
  .widget-upload,.widget-button{width:100%!important;min-height:52px!important;font-size:16px!important;}
  .widget-toggle-buttons .widget-toggle-button{font-size:11px!important;padding:4px 6px!important;}
}
</style>
"""))

# ════════════════════════════════════════════════
# §3  Інтерфейс
# ════════════════════════════════════════════════
_title = widgets.HTML(f"""
<div style="font-family:sans-serif;">
  <h2 style="margin:0 0 4px;">🎨 PhotoEnhance MVP</h2>
  <p style="color:#666;margin:0;">Real-ESRGAN + CodeFormer | Ліміт: <b>{MAX_FILE_MB} MB</b> | JPG/PNG/WEBP</p>
</div>
""")

upload_btn = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.webp',
    multiple=False,
    description='📸 Вибрати фото',
    layout=widgets.Layout(width='auto', min_width='200px', min_height='44px'),
)

photo_type = widgets.ToggleButtons(
    options=[
        ('🧑 Портрет',          'portrait'),
        ('👟 Товар / Текстура', 'product'),
        ('🌄 Пейзаж',          'landscape'),
        ('📷 Старе фото',       'old_photo'),
        ('🎌 Аніме',            'anime'),
    ],
    value='product',
    description='Тип:',
    style={'button_width': 'auto', 'description_width': '50px'},
    layout=widgets.Layout(width='100%'),
)

_type_desc = widgets.HTML("""
<div style="font-family:sans-serif;font-size:12px;color:#1a1a2e;
            background:#e8f0fe;border-left:4px solid #4a90e2;
            border-radius:6px;padding:10px 14px;margin-top:4px;line-height:1.7;">
  <b>🧑 Портрет</b> — CodeFormer: відновлює обличчя, зберігає природну шкіру.<br>
  <b>👟 Товар</b> — Real-ESRGAN: різкі деталі тканини, шкіри, гуми.<br>
  <b>🌄 Пейзаж</b> — Real-ESRGAN: чіткі контури та деталі фону.<br>
  <b>📷 Старе фото</b> — CodeFormer (обличчя) + ESRGAN (фон): усуває деградації.<br>
  <b>🎌 Аніме</b> — anime-6B модель: чисті лінії та кольори.
</div>
""")

face_fidelity = widgets.FloatSlider(
    value=0.8, min=0.0, max=1.0, step=0.05,
    description='Fidelity:',
    readout_format='.2f',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='100%', max_width='480px'),
)
_fid_desc = widgets.HTML(
    '<div style="font-size:11px;color:#666;margin-top:-6px;">'
    '0.0 = макс відновлення &nbsp;|&nbsp; 0.8 = природна шкіра ✅ &nbsp;|&nbsp; 1.0 = майже без змін'
    '</div>'
)
_fid_box = widgets.VBox([face_fidelity, _fid_desc])
_fid_box.layout.display = 'none'

def _on_type_change(change):
    _fid_box.layout.display = '' if change['new'] in ('portrait', 'old_photo') else 'none'

photo_type.observe(_on_type_change, names='value')

scale_options = widgets.ToggleButtons(
    options=[('×2  (швидше)', 2), ('×4  (оптимально)', 4), ('×8  (max розмір)', 8)],
    value=4,
    description='Масштаб:',
    style={'button_width': 'auto', 'description_width': '70px'},
)

model_options = widgets.Dropdown(
    options=[
        ('🖼️  Загальні фото',          'RealESRGAN_x4plus'),
        ('🎌 Аніме / Ілюстрації',       'RealESRGAN_x4plus_anime_6B'),
        ('📷 Деградовані фото',          'ESRGAN_SRx4_DF2KOST'),
        ('⚡ Швидкий режим (×2)',        'RealESRGAN_x2plus'),
    ],
    value='RealESRGAN_x4plus',
    description='Модель:',
    layout=widgets.Layout(width='100%', max_width='440px'),
)
_model_note = widgets.HTML(
    '<div style="font-size:11px;color:#888;">⚠️ Для Портрет / Старе фото / Аніме — модель обирається автоматично.</div>'
)

_preview = widgets.Output()
_info    = widgets.HTML('<div class="pe-info">📁 Фото ще не вибрано</div>')

def _on_upload(*_):
    """Викликається при виборі файлу (observer callback, аргумент не потрібен)."""
    if not upload_btn.value:
        return
    for _fname, _fdata in upload_btn.value.items():
        _size_mb = len(_fdata['content']) / 1048576
        if _size_mb > MAX_FILE_MB:
            _info.value = (
                f'<div class="pe-error">❌ <b>Файл {_size_mb:.1f} MB</b> — '
                f'перевищує ліміт {MAX_FILE_MB} MB. Стисни або обери інший.</div>'
            )
            with _preview:
                clear_output()
            return

        try:
            _img = Image.open(io.BytesIO(_fdata['content']))
        except Exception as _e:
            _info.value = f'<div class="pe-error">❌ Неможливо відкрити файл: {_e}</div>'
            with _preview:
                clear_output()
            return

        _w, _h = _img.size
        _mpx   = _w * _h / 1_000_000
        _info.value = (
            f'<div class="pe-info">'
            f'✅ <b>{_fname}</b><br>'
            f'📐 {_w}×{_h} px &nbsp;|&nbsp; {_size_mb:.1f} MB &nbsp;|&nbsp;'
            f' {_img.mode} &nbsp;|&nbsp; {_mpx:.1f} Mpx'
            f'</div>'
        )
        with _preview:
            clear_output(wait=True)
            _thumb = _img.copy()
            _thumb.thumbnail((280, 280))
            _, _ax = plt.subplots(figsize=(3.5, 3.5))
            _ax.imshow(_thumb)
            _ax.set_title('Прев\'ю оригіналу', fontsize=10)
            _ax.axis('off')
            plt.tight_layout()
            plt.show()

upload_btn.observe(_on_upload, names='value')

_ui = widgets.VBox([
    _title,
    widgets.HTML('<hr style="margin:8px 0">'),
    upload_btn, _info, _preview,
    widgets.HTML('<hr style="margin:8px 0">'),
    photo_type, _type_desc, _fid_box,
    widgets.HTML('<hr style="margin:8px 0">'),
    model_options, _model_note, scale_options,
    widgets.HTML('<hr style="margin:8px 0">'),



    widgets.HTML('<b>✅ Готово? Запускай ▶ Крок 3</b>'),
], layout=widgets.Layout(width='100%', max_width='660px'))
display(_ui)

In [ ]:
    up.model.eval()

    # NOTE: torch.compile з mode='reduce-overhead' використовує CUDAGraphs,
    # що конфліктує з тайловим циклом (кожен тайл перезаписує попередній output).
    # Для ESRGAN compile не застосовуємо — він і так швидкий через FP16.

    _dev = torch.device('cuda' if USE_FP16 else 'cpu')
    _warmup(up.model, _dev, USE_FP16)

In [ ]:
# @title 🔍 ДІАГНОСТИКА — запусти та скопіюй весь вивід
import subprocess, sys, os

print("=" * 60)
print("1) basicsr версія та шлях:")
r = subprocess.run(['pip', 'show', 'basicsr'], capture_output=True, text=True)
print(r.stdout.strip())

print("\n2) realesrgan версія та шлях:")
r2 = subprocess.run(['pip', 'show', 'realesrgan'], capture_output=True, text=True)
print(r2.stdout.strip())

print("\n3) Де Python знаходить basicsr:")
try:
    import basicsr
    bpath = os.path.dirname(basicsr.__file__)
    print(f"  {bpath}")
    print("\n4) Файли у basicsr/data/:")
    data_path = os.path.join(bpath, 'data')
    if os.path.isdir(data_path):
        for f in sorted(os.listdir(data_path)):
            print(f"  {f}")
    else:
        print("  ⚠️  Папка basicsr/data НЕ існує!")
except Exception as e:
    print(f"  ❌ import basicsr: {e}")

print("\n5) sys.path (перші 10):")
for p in sys.path[:10]:
    print(f"  {p}")

print("\n6) Де Python знаходить realesrgan:")
try:
    import importlib, importlib.util
    spec = importlib.util.find_spec('realesrgan')
    print(f"  {spec.origin if spec else 'не знайдено'}")
    print("\n7) Вміст realesrgan/__init__.py (перші 30 рядків):")
    init = os.path.join(os.path.dirname(spec.origin), '__init__.py') if spec else None
    if init and os.path.isfile(init):
        with open(init) as f:
            for i, line in enumerate(f):
                if i >= 30: break
                print(f"  {line}", end='')
    else:
        print("  не знайдено")
except Exception as e:
    print(f"  ❌ {e}")

print("\n" + "=" * 60)
print("Скопіюй весь цей вивід — це дасть точну відповідь!")
